# Prototipo tésis

# Subtarea 1

In [6]:
# ======================================================
#  VALIDACIÓN CRUZADA (5-FOLD)
#  Subtask 1: Polarización Binaria
# ======================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import RandomOverSampler
from lightgbm import LGBMClassifier

import nltk
nltk.download("punkt")
nltk.download('punkt_tab')
nltk.download("stopwords")

from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from scipy.sparse import hstack, csr_matrix
from scipy.stats import ttest_rel   # 👈 agregado

# ======================================================
# CONFIG
# ======================================================

RANDOM_STATE = 42
N_SPLITS = 5
EMBED_MODEL = "sentence-transformers/distiluse-base-multilingual-cased-v2"
K = 3
MAX_FEATURES = 5000

# ======================================================
# DATA
# ======================================================

df = pd.read_csv("Unified_Dataset_Clean_Final.csv")
df = df[df["lang"].isin(["spa", "eng"])].copy()
df = df.dropna(subset=["text", "polarization"])

texts = df["text"].astype(str).tolist()
y = df["polarization"].astype(int).values

print("Dataset:", df.shape)

# ======================================================
# EMBEDDINGS
# ======================================================

print("🔹 Generando embeddings...")
embedder = SentenceTransformer(EMBED_MODEL)
X_embed = embedder.encode(texts, show_progress_bar=True)
X_embed_sparse = csr_matrix(X_embed)

# ======================================================
# k-MERS
# ======================================================

def generate_kmers(text, k=3):
    tokens = word_tokenize(text.lower())
    if len(tokens) < k:
        return text.lower()
    kmers = [
        "_".join(tokens[i:i+k])
        for i in range(len(tokens) - k + 1)
    ]
    return " ".join(kmers)

print("🔹 Generando k-mers...")
texts_kmers = [generate_kmers(t, K) for t in texts]

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=2
)

X_kmers = vectorizer.fit_transform(texts_kmers)

# ======================================================
# REPRESENTACIONES
# ======================================================

X_llm = X_embed_sparse
X_genoma = X_kmers
X_hybrid = hstack([X_embed_sparse, X_kmers])

representations = {
    "LLMeval (Embeddings)": X_llm,
    "Genoma-kmers": X_genoma,
    "POLAR+Genoma": X_hybrid
}

# ======================================================
# MODELOS
# ======================================================

models = {
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "SVM": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

# ======================================================
# EVALUACIÓN
# ======================================================

def eval_kfold_model(X, y, base_model):

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):

        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        ros = RandomOverSampler(random_state=RANDOM_STATE)
        X_tr, y_tr = ros.fit_resample(X_tr, y_tr)

        clf = clone(base_model)
        clf.fit(X_tr, y_tr)

        y_pred = clf.predict(X_te)
        f1 = f1_score(y_te, y_pred, average="macro")

        scores.append(f1)
        print(f"  Fold {fold}: F1(macro) = {f1:.4f}")

    scores = np.array(scores)
    mean = scores.mean()
    std = scores.std(ddof=1)

    ci_low = mean - 1.96 * (std / np.sqrt(N_SPLITS))
    ci_high = mean + 1.96 * (std / np.sqrt(N_SPLITS))

    return mean, std, ci_low, ci_high, scores  # 👈 devolvemos folds

# ======================================================
# EJECUCIÓN
# ======================================================

results = []
fold_storage = {}  # 👈 guardamos folds aquí

for rep_name, X_rep in representations.items():

    print("\n=================================================")
    print("Representación:", rep_name)
    print("=================================================")

    for model_name, model in models.items():

        print("\nModelo:", model_name)

        mean, std, ci_low, ci_high, fold_scores = eval_kfold_model(
            X_rep, y, model
        )

        key = f"{rep_name} | {model_name}"
        fold_storage[key] = fold_scores  # 👈 guardamos folds

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "F1_mean": mean,
            "Std": std,
            "CI_low": ci_low,
            "CI_high": ci_high
        })

# ==============================
# TABLA 1: RESULTADOS GENERALES
# ==============================

df_results = pd.DataFrame(results)

print("\n===== TABLA 1: RESULTADOS GENERALES =====\n")
print(df_results.sort_values("F1_mean", ascending=False))

# ==============================
# TABLA 2: PRUEBA ESTADÍSTICA
# ==============================

comparisons = []

for model_name in models.keys():

    key_llm = f"LLMeval (Embeddings) | {model_name}"
    key_gen = f"Genoma-kmers | {model_name}"
    key_hybrid = f"POLAR+Genoma | {model_name}"

    if key_llm in fold_storage and key_hybrid in fold_storage:
        t_stat, p_val = ttest_rel(
            fold_storage[key_llm],
            fold_storage[key_hybrid]
        )

        comparisons.append({
            "Modelo": model_name,
            "Comparación": "LLMeval vs POLAR+Genoma",
            "t_stat": t_stat,
            "p_value": p_val,
            "Significativo (p<0.05)": p_val < 0.05
        })

df_stats = pd.DataFrame(comparisons)

print("\n===== TABLA 2: PRUEBA ESTADÍSTICA (t-test pareado) =====\n")
print(df_stats)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dataset: (5237, 15)
🔹 Generando embeddings...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/164 [00:00<?, ?it/s]

🔹 Generando k-mers...

Representación: LLMeval (Embeddings)

Modelo: LightGBM
[LightGBM] [Info] Number of positive: 3111, number of negative: 3111
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035114 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130560
[LightGBM] [Info] Number of data points in the train set: 6222, number of used features: 512
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


...

  Fold 3: F1(macro) = 0.7824
[LightGBM] [Info] Number of positive: 3111, number of negative: 3111
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.068121 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 139492
[LightGBM] [Info] Number of data points in the train set: 6222, number of used features: 959
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 4: F1(macro) = 0.7535
[LightGBM] [Info] Number of positive: 3112, number of negative: 3112
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.067104 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 139481
[LightGBM] [Info] Number of data points in the train set: 6224, number of used features: 949
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 5: F1(macro) = 0.7956

Modelo: Logistic Regression
  Fold 1: F1(macro) = 0.7490
  Fold 2: F1(macro) = 0.7764
  Fold 3: F1(macro) = 0.7889
  Fold 4: F1(macro) = 0.7696
  Fold 5: F1(macro) = 0.7725

Modelo: SVM
  Fold 1: F1(macro) = 0.7512
  Fold 2: F1(macro) = 0.7684
  Fold 3: F1(macro) = 0.7662
  Fold 4: F1(macro) = 0.7764
  Fold 5: F1(macro) = 0.7782

===== TABLA 1: RESULTADOS GENERALES =====

         Representación               Modelo   F1_mean       Std    CI_low  \
6          POLAR+Genoma             LightGBM  0.778256  0.020428  0.760350   
7          POLAR+Genoma  Logistic Regression  0.771282  0.014476  0.758593   
8          POLAR+Genoma                  SVM  0.768072  0.010751  0.758647   
0  LLMeval (Embeddings)             LightGBM  0.756473  0.018220  0.740502   
2  LLMeval (Embeddings)                  SVM  0.736839  0.011597  0.726674   
1  LLMeval (Embeddings)  Logistic Regression  0.731076  0.019219  0.714231   
4          Genoma-kmers  Logistic Regression  0.7

# Subtarea 2

In [8]:
# ======================================================
#  SUBTASK 2: Polarization Target Classification (Multiclass)
#  VERSION OPTIMIZADA
# ======================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer
from nltk.tokenize import word_tokenize
from scipy.sparse import hstack, csr_matrix
from scipy.stats import ttest_rel

import nltk
nltk.download("punkt")

# ======================================================
# CONFIG
# ======================================================

RANDOM_STATE = 42
N_SPLITS = 5
EMBED_MODEL = "sentence-transformers/distiluse-base-multilingual-cased-v2"
K = 3
MAX_FEATURES = 3000   # 🔥 reducido

# ======================================================
# DATA
# ======================================================

df = pd.read_csv("Unified_Dataset_Clean_Final.csv")
df = df[df["lang"].isin(["spa", "eng"])].dropna(subset=["text"])

target_cols = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
df["target_label"] = df[target_cols].idxmax(axis=1)

texts = df["text"].astype(str).tolist()
y = df["target_label"].astype("category").cat.codes.values

# ======================================================
# EMBEDDINGS
# ======================================================

print("🔹 Generando embeddings...")
embedder = SentenceTransformer(EMBED_MODEL)
X_embed = embedder.encode(texts, show_progress_bar=True)
X_embed_sparse = csr_matrix(X_embed)

# ======================================================
# k-MERS
# ======================================================

def generate_kmers(text, k=3):
    tokens = word_tokenize(text.lower())
    if len(tokens) < k:
        return text.lower()
    kmers = [
        "_".join(tokens[i:i+k])
        for i in range(len(tokens) - k + 1)
    ]
    return " ".join(kmers)

print("🔹 Generando k-mers...")
texts_kmers = [generate_kmers(t, K) for t in texts]

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=2
)

X_kmers = vectorizer.fit_transform(texts_kmers)

# ======================================================
# REPRESENTACIONES
# ======================================================

X_llm = X_embed_sparse
X_genoma = X_kmers
X_hybrid = hstack([X_embed_sparse, X_kmers])

representations = {
    "LLMeval": X_llm,
    "Genoma-kmers": X_genoma,
    "LLMeval+Genoma": X_hybrid
}

# ======================================================
# MODELOS
# ======================================================

models = {
    "LightGBM": LGBMClassifier(
        n_estimators=150,   # 🔥 reducido
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        n_jobs=-1           # 🔥 paralelización
    ),
    "Logistic": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        multi_class="ovr",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "SVM": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

# ======================================================
# EVALUACIÓN
# ======================================================

def eval_kfold_model(X, y, base_model):

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):

        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        clf = clone(base_model)
        clf.fit(X_tr, y_tr)

        y_pred = clf.predict(X_te)

        f1 = f1_score(y_te, y_pred, average="macro", zero_division=0)
        scores.append(f1)

        print(f"  Fold {fold}: F1(macro) = {f1:.4f}")

    scores = np.array(scores)
    mean = scores.mean()
    std = scores.std(ddof=1)

    ci_low = mean - 1.96 * (std / np.sqrt(N_SPLITS))
    ci_high = mean + 1.96 * (std / np.sqrt(N_SPLITS))

    return mean, std, ci_low, ci_high, scores

# ======================================================
# EJECUCIÓN
# ======================================================

results = []
fold_storage = {}

for rep_name, X_rep in representations.items():

    print("\n=================================================")
    print("Representación:", rep_name)
    print("=================================================")

    for model_name, model in models.items():

        print("\nModelo:", model_name)

        mean, std, ci_low, ci_high, fold_scores = eval_kfold_model(
            X_rep, y, model
        )

        key = f"{rep_name} | {model_name}"
        fold_storage[key] = fold_scores

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "F1_mean": mean,
            "Std": std,
            "CI_low": ci_low,
            "CI_high": ci_high
        })

# ======================================================
# TABLA 1
# ======================================================

df_results = pd.DataFrame(results)

print("\n===== TABLA 1: RESULTADOS GENERALES =====\n")
print(df_results.sort_values("F1_mean", ascending=False))

# ======================================================
# TABLA 2 – t-test
# ======================================================

comparisons = []

for model_name in models.keys():

    key_llm = f"LLMeval | {model_name}"
    key_hybrid = f"LLMeval+Genoma | {model_name}"

    if key_llm in fold_storage and key_hybrid in fold_storage:

        t_stat, p_val = ttest_rel(
            fold_storage[key_llm],
            fold_storage[key_hybrid]
        )

        comparisons.append({
            "Modelo": model_name,
            "Comparación": "LLMeval vs LLMeval+Genoma",
            "t_stat": t_stat,
            "p_value": p_val,
            "Significativo (p<0.05)": p_val < 0.05
        })

df_stats = pd.DataFrame(comparisons)

print("\n===== TABLA 2: PRUEBA ESTADÍSTICA =====\n")
print(df_stats)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


🔹 Generando embeddings...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/164 [00:00<?, ?it/s]

🔹 Generando k-mers...

Representación: LLMeval

Modelo: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.078058 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130560
[LightGBM] [Info] Number of data points in the train set: 4189, number of used features: 512
[LightGBM] [Info] Start training from score -3.686257
[LightGBM] [Info] Start training from score -2.073017
[LightGBM] [Info] Start training from score -0.183134
[LightGBM] [Info] Start training from score -4.702631
[LightGBM] [Info] Start training from score -4.906230
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


...

  Fold 3: F1(macro) = 0.5953


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


  Fold 4: F1(macro) = 0.5925


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


  Fold 5: F1(macro) = 0.5636

Modelo: SVM
  Fold 1: F1(macro) = 0.5591
  Fold 2: F1(macro) = 0.6721
  Fold 3: F1(macro) = 0.6636
  Fold 4: F1(macro) = 0.5695
  Fold 5: F1(macro) = 0.5751

===== TABLA 1: RESULTADOS GENERALES =====

   Representación    Modelo   F1_mean       Std    CI_low   CI_high
8  LLMeval+Genoma       SVM  0.607890  0.055123  0.559572  0.656207
2         LLMeval       SVM  0.604477  0.027860  0.580057  0.628898
7  LLMeval+Genoma  Logistic  0.595434  0.027206  0.571587  0.619280
1         LLMeval  Logistic  0.560558  0.034619  0.530213  0.590903
5    Genoma-kmers       SVM  0.410924  0.018441  0.394760  0.427088
6  LLMeval+Genoma  LightGBM  0.383903  0.037153  0.351337  0.416470
4    Genoma-kmers  Logistic  0.377674  0.017426  0.362399  0.392948
0         LLMeval  LightGBM  0.369741  0.053829  0.322558  0.416925
3    Genoma-kmers  LightGBM  0.207625  0.018331  0.191557  0.223693

===== TABLA 2: PRUEBA ESTADÍSTICA =====

     Modelo                Comparación    t_sta

# Subtarea 3

In [9]:
# ======================================================
#  SUBTASK 3: Linguistic Manifestation (Multilabel)
#  Representaciones:
#     - LLMeval (Embeddings)
#     - Genoma-kmers (TF-IDF k=3)
#     - LLMeval+Genoma (Híbrido)
#  Modelos:
#     - LightGBM
#     - Logistic
#     - SVM
#  Incluye:
#     - Std
#     - IC 95%
#     - t-test pareado
# ======================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier

from lightgbm import LGBMClassifier
from sentence_transformers import SentenceTransformer
from scipy.sparse import hstack, csr_matrix
from scipy.stats import ttest_rel
from nltk.tokenize import word_tokenize

import nltk
nltk.download("punkt")

# ======================================================
# CONFIG
# ======================================================

RANDOM_STATE = 42
N_SPLITS = 5
EMBED_MODEL = "sentence-transformers/distiluse-base-multilingual-cased-v2"
K = 3
MAX_FEATURES = 5000

# ======================================================
# DATA
# ======================================================

df = pd.read_csv("Unified_Dataset_Clean_Final.csv")
df = df[df["lang"].isin(["spa", "eng"])].dropna(subset=["text"])

targets = [
    "vilification",
    "extreme_language",
    "stereotype",
    "invalidation",
    "lack_of_empathy",
    "dehumanization"
]

df[targets] = df[targets].fillna(0).astype(int)

texts = df["text"].astype(str).tolist()
Y = df[targets].values

# Estratificación auxiliar
y_aux = df["polarization"].astype(int).values

# ======================================================
# EMBEDDINGS
# ======================================================

print("🔹 Generando embeddings...")
embedder = SentenceTransformer(EMBED_MODEL)
X_embed = embedder.encode(texts, show_progress_bar=True)
X_embed_sparse = csr_matrix(X_embed)

# ======================================================
# k-MERS
# ======================================================

def generate_kmers(text, k=3):
    tokens = word_tokenize(text.lower())
    if len(tokens) < k:
        return text.lower()
    kmers = [
        "_".join(tokens[i:i+k])
        for i in range(len(tokens) - k + 1)
    ]
    return " ".join(kmers)

print("🔹 Generando k-mers...")
texts_kmers = [generate_kmers(t, K) for t in texts]

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=2
)

X_kmers = vectorizer.fit_transform(texts_kmers)

# ======================================================
# REPRESENTACIONES
# ======================================================

X_llm = X_embed_sparse
X_genoma = X_kmers
X_hybrid = hstack([X_embed_sparse, X_kmers])

representations = {
    "LLMeval": X_llm,
    "Genoma-kmers": X_genoma,
    "LLMeval+Genoma": X_hybrid
}

# ======================================================
# MODELOS
# ======================================================

models = {
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE
    ),
    "Logistic": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "SVM": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

# ======================================================
# EVALUACIÓN
# ======================================================

def eval_kfold_model(X, Y, base_model):

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y_aux), 1):

        X_tr, X_te = X[tr_idx], X[te_idx]
        Y_tr, Y_te = Y[tr_idx], Y[te_idx]

        clf = MultiOutputClassifier(clone(base_model))
        clf.fit(X_tr, Y_tr)

        Y_pred = clf.predict(X_te)

        f1 = f1_score(Y_te, Y_pred, average="macro", zero_division=0)
        scores.append(f1)

        print(f"  Fold {fold}: F1(macro) = {f1:.4f}")

    scores = np.array(scores)
    mean = scores.mean()
    std = scores.std(ddof=1)

    ci_low = mean - 1.96 * (std / np.sqrt(N_SPLITS))
    ci_high = mean + 1.96 * (std / np.sqrt(N_SPLITS))

    return mean, std, ci_low, ci_high, scores

# ======================================================
# EJECUCIÓN
# ======================================================

results = []
fold_storage = {}

for rep_name, X_rep in representations.items():

    print("\n=================================================")
    print("Representación:", rep_name)
    print("=================================================")

    for model_name, model in models.items():

        print("\nModelo:", model_name)

        mean, std, ci_low, ci_high, fold_scores = eval_kfold_model(
            X_rep, Y, model
        )

        key = f"{rep_name} | {model_name}"
        fold_storage[key] = fold_scores

        results.append({
            "Representación": rep_name,
            "Modelo": model_name,
            "F1_mean": mean,
            "Std": std,
            "CI_low": ci_low,
            "CI_high": ci_high
        })

# ======================================================
# TABLA 1
# ======================================================

df_results = pd.DataFrame(results)

print("\n===== TABLA 1: RESULTADOS GENERALES =====\n")
print(df_results.sort_values("F1_mean", ascending=False))

# ======================================================
# TABLA 2 – t-test
# ======================================================

comparisons = []

for model_name in models.keys():

    key_llm = f"LLMeval | {model_name}"
    key_hybrid = f"LLMeval+Genoma | {model_name}"

    if key_llm in fold_storage and key_hybrid in fold_storage:

        t_stat, p_val = ttest_rel(
            fold_storage[key_llm],
            fold_storage[key_hybrid]
        )

        comparisons.append({
            "Modelo": model_name,
            "Comparación": "LLMeval vs LLMeval+Genoma",
            "t_stat": t_stat,
            "p_value": p_val,
            "Significativo (p<0.05)": p_val < 0.05
        })

df_stats = pd.DataFrame(comparisons)

print("\n===== TABLA 2: PRUEBA ESTADÍSTICA =====\n")
print(df_stats)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


🔹 Generando embeddings...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/164 [00:00<?, ?it/s]

🔹 Generando k-mers...

Representación: LLMeval

Modelo: LightGBM
[LightGBM] [Info] Number of positive: 259, number of negative: 3930
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.061272 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130560
[LightGBM] [Info] Number of data points in the train set: 4189, number of used features: 512
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.061829 -> initscore=-2.719567
[LightGBM] [Info] Start training from score -2.719567
[LightGBM] [Info] Number of positive: 289, number of negative: 3900
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025849 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 130560
[LightGBM] [Info] Number of data points in the train set: 4189, number of used features: 512
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.068990 -> initscore=-2.602305
[Li

...

  Fold 3: F1(macro) = 0.1795
[LightGBM] [Info] Number of positive: 268, number of negative: 3922
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071786 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 136937
[LightGBM] [Info] Number of data points in the train set: 4190, number of used features: 778
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.063962 -> initscore=-2.683370
[LightGBM] [Info] Start training from score -2.683370
[LightGBM] [Info] Number of positive: 280, number of negative: 3910
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042898 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 136937
[LightGBM] [Info] Number of data points in the train set: 4190, number of used features: 778
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.066826 -> initscore=-2.636503
[LightGBM] [Info] Start training from s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

  Fold 4: F1(macro) = 0.3413
[LightGBM] [Info] Number of positive: 259, number of negative: 3931
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.041616 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 136981
[LightGBM] [Info] Number of data points in the train set: 4190, number of used features: 782
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.061814 -> initscore=-2.719821
[LightGBM] [Info] Start training from score -2.719821
[LightGBM] [Info] Number of positive: 289, number of negative: 3901
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.047593 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 136981
[LightGBM] [Info] Number of data points in the train set: 4190, number of used features: 782
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.068974 -> initscore=-2.602562
[LightGBM] [Info] Start training from s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

  Fold 5: F1(macro) = 0.2780

Modelo: Logistic
  Fold 1: F1(macro) = 0.2434
  Fold 2: F1(macro) = 0.3270
  Fold 3: F1(macro) = 0.2427
  Fold 4: F1(macro) = 0.3635
  Fold 5: F1(macro) = 0.3080

Modelo: SVM
  Fold 1: F1(macro) = 0.2647
  Fold 2: F1(macro) = 0.3648
  Fold 3: F1(macro) = 0.2525
  Fold 4: F1(macro) = 0.4894
  Fold 5: F1(macro) = 0.3596

===== TABLA 1: RESULTADOS GENERALES =====

   Representación    Modelo   F1_mean       Std    CI_low   CI_high
8  LLMeval+Genoma       SVM  0.346217  0.095456  0.262546  0.429888
2         LLMeval       SVM  0.310629  0.078069  0.242199  0.379059
7  LLMeval+Genoma  Logistic  0.296922  0.053069  0.250405  0.343439
1         LLMeval  Logistic  0.279546  0.068917  0.219138  0.339954
6  LLMeval+Genoma  LightGBM  0.254815  0.065605  0.197310  0.312321
0         LLMeval  LightGBM  0.253634  0.067455  0.194508  0.312761
4    Genoma-kmers  Logistic  0.176017  0.039329  0.141544  0.210491
5    Genoma-kmers       SVM  0.157790  0.021289  0.139130  0.1